In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
BASE_PATH = Path('.')

MODEL_NAME = "unsloth/gemma-4-E4B-it"
MODEL_NAME_SLUG = MODEL_NAME.split("/")[-1].lower().replace("-", "_")

PREDICTIONS_DIR = BASE_PATH / f"{MODEL_NAME_SLUG}_predictions"
PRED_BASE = PREDICTIONS_DIR / f"{MODEL_NAME_SLUG}_predictions_base.json"
PRED_TUNED = PREDICTIONS_DIR / f"{MODEL_NAME_SLUG}_predictions_tuned.json"

METRICS_DIR = BASE_PATH / f"{MODEL_NAME_SLUG}_metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

OUT_BASE_METRICS_JSON = METRICS_DIR / f"{MODEL_NAME_SLUG}_metrics_base.json"
OUT_TUNED_METRICS_JSON = METRICS_DIR / f"{MODEL_NAME_SLUG}_metrics_tuned.json"
OUT_COMBINED_CSV = METRICS_DIR / f"{MODEL_NAME_SLUG}_metrics_combined.csv"

PLOTS_DIR = METRICS_DIR / f"{MODEL_NAME_SLUG}_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

PRED_BASE, PRED_TUNED, PRED_BASE.exists(), PRED_TUNED.exists()


In [ ]:
def read_predictions(path):
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list):
        raise ValueError(f'Expected list in {path}, got {type(data).__name__}')
    return data


def extract_keys(rows):
    keys = set()
    for r in rows:
        gold = r.get('gold')
        if isinstance(gold, dict):
            keys.update(map(str, gold.keys()))
    return sorted(keys)


In [ ]:
def compute_scores(y_true, y_pred):
    if not y_true:
        return {'accuracy': 0.0, 'f1_micro': 0.0, 'f1_macro': 0.0, 'f1_weighted': 0.0}

    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_micro': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }


def compute_metrics(rows, tag):
    keys = extract_keys(rows)
    if not keys:
        raise ValueError('No keys found in gold labels; need `gold` in predictions JSON')

    per_key = {}

    y_true_all = []
    y_pred_all = []

    for k in keys:
        y_true = []
        y_pred = []

        for r in rows:
            gold = r.get('gold')
            pred = r.get('pred')
            if not isinstance(gold, dict):
                continue

            g = str(gold.get(k, ''))
            p = '' if not isinstance(pred, dict) else str(pred.get(k, ''))
            y_true.append(g)
            y_pred.append(p)

        per_key[k] = compute_scores(y_true, y_pred)
        y_true_all.extend(y_true)
        y_pred_all.extend(y_pred)

    overall = compute_scores(y_true_all, y_pred_all)

    return {
        'tag': tag,
        'n_rows': len(rows),
        'n_with_gold': sum(1 for r in rows if isinstance(r.get('gold'), dict)),
        'n_unparsed': sum(1 for r in rows if r.get('pred') is None),
        'overall': overall,
        'per_key': per_key,
    }


In [ ]:
def metrics_to_frames(metrics):
    tag = str(metrics['tag'])

    overall_row = {'tag': tag, 'key': '__overall__', **metrics['overall']}
    overall_df = pd.DataFrame([overall_row])

    per_key_rows = [{'tag': tag, 'key': k, **v} for k, v in metrics['per_key'].items()]
    per_key_df = pd.DataFrame(per_key_rows)

    cols = ['tag', 'key', 'accuracy', 'f1_micro', 'f1_macro', 'f1_weighted']
    return overall_df[cols], per_key_df[cols]


def save_metrics(metrics, out_json):
    out_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
base_rows = read_predictions(PRED_BASE)
tuned_rows = read_predictions(PRED_TUNED)

base_metrics = compute_metrics(base_rows, tag='base')
tuned_metrics = compute_metrics(tuned_rows, tag='tuned')

save_metrics(base_metrics, OUT_BASE_METRICS_JSON)
save_metrics(tuned_metrics, OUT_TUNED_METRICS_JSON)

base_overall_df, base_per_key_df = metrics_to_frames(base_metrics)
tuned_overall_df, tuned_per_key_df = metrics_to_frames(tuned_metrics)

combined_df = pd.concat(
    [base_overall_df, tuned_overall_df, base_per_key_df, tuned_per_key_df],
    ignore_index=True,
)
combined_df.to_csv(OUT_COMBINED_CSV, index=False)

combined_df


In [ ]:
def plot_metric_bars(per_key_df, metric, out_path, model_name):
    pivot = per_key_df.pivot_table(index='key', columns='tag', values=metric, aggfunc='mean')
    pivot = pivot.sort_values(by=pivot.columns[0], ascending=False)

    keys = list(pivot.index)
    x = list(range(1, len(keys) + 1))

    fig = plt.figure(figsize=(16, 7.5))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.4, 6.1])

    ax_text = fig.add_subplot(gs[0])
    ax = fig.add_subplot(gs[1])

    width = 0.38

    cols = list(pivot.columns)
    if len(cols) == 1:
        cols = cols + [cols[0]]

    ax.bar([v - width / 2 for v in x], pivot[cols[0]].values, width=width, label=str(cols[0]))
    ax.bar([v + width / 2 for v in x], pivot[cols[1]].values, width=width, label=str(cols[1]))

    ax.set_xticks(x)
    ax.set_xticklabels([str(i) for i in x], rotation=0)

    ax.set_title(f"{model_name}: {metric} (base vs tuned)", pad=8)
    ax.set_xlabel('Признак')
    ax.set_ylabel(metric)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.legend(title='tag', loc='upper right')

    mapping_lines = [f"{i}. {k}" for i, k in enumerate(keys, start=1)]
    mapping_text = "\n".join(mapping_lines)

    ax_text.axis('off')
    ax_text.text(
        0.0,
        1.0,
        mapping_text,
        ha='left',
        va='top',
        fontsize=9,
        family='monospace',
        transform=ax_text.transAxes,
    )

    fig.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close(fig)


def plot_overall_lines(overall_df, out_path, model_name):
    overall_long = overall_df.melt(
        id_vars=['tag', 'key'],
        value_vars=['accuracy', 'f1_micro', 'f1_macro', 'f1_weighted'],
        var_name='metric',
        value_name='value',
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    for tag in sorted(overall_long['tag'].unique()):
        sub = overall_long[overall_long['tag'] == tag]
        ax.plot(sub['metric'], sub['value'], marker='o', label=tag)

    ax.set_title(f"{model_name}: overall (base vs tuned)")
    ax.set_xlabel('Метрика')
    ax.set_ylabel('Значение')
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(title='tag')
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close(fig)


per_key_df = combined_df[combined_df['key'] != '__overall__'].copy()
overall_df = combined_df[combined_df['key'] == '__overall__'].copy()

model_name = MODEL_NAME

for m in ['accuracy', 'f1_micro', 'f1_macro', 'f1_weighted']:
    plot_metric_bars(per_key_df, metric=m, out_path=PLOTS_DIR / f'per_key_{m}.png', model_name=model_name)

plot_overall_lines(overall_df, out_path=PLOTS_DIR / 'overall_metrics.png', model_name=model_name)

sorted([p.name for p in PLOTS_DIR.glob('*.png')]), overall_df
